## A short notebook tutorial on Spherical Elementary Currents (Ridge) Regression

*Last Modified*: July 24th, 2024

Author: [Opal Issan](https://opaliss.github.io/opalissan/) (PhD student @UCSD). contact: oissan@ucsd.edu

In [ ]:
import sys, os

sys.path.append(os.path.abspath(os.path.join("..")))

In [ ]:
import numpy as np
import scipy
import cartopy.crs as ccrs
from supermag_api import *
from sec import T_df
import stripy
from spherical_harmonics import ridge_regression

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from mycolorpy import colorlist as mcp

font = {"family": "serif", "size": 14}

matplotlib.rc("font", **font)
matplotlib.rc("xtick", labelsize=14)
matplotlib.rc("ytick", labelsize=14)

# SuperMAG data

In [ ]:
# start date year-month-day-hour-min-sec
start = [2024, 2, 17, 18, 40, 10]

In [ ]:
# read in data
status, stations = SuperMAGGetInventory("opaliss", start, 3600)
# number of stations
N = len(stations)
print("number of stations = ", N)

In [ ]:
# intialize data
data_Bn = np.zeros(N)
data_Be = np.zeros(N)
data_Bz = np.zeros(N)
geo_lat = np.zeros(N)
geo_lon = np.zeros(N)

In [ ]:
# read in data for 1hr in advance from start date at station "res"
# note: this is very slow.. not sure if there are faster ways to go about this
kk = 0
for ii in range(0, N):
    status, sm_data = SuperMAGGetData("opaliss", start, 3600, "geo", stations[ii])
    try:
        if sm_data.glat[0] not in geo_lat and sm_data.glon[0] not in geo_lon:
            data_Bn[kk] = sm_data.N[0]["geo"]
            data_Be[kk] = sm_data.E[0]["geo"]
            data_Bz[kk] = sm_data.Z[0]["geo"]
            geo_lat[kk] = sm_data.glat[0]
            geo_lon[kk] = sm_data.glon[0]
            kk += 1
    except:
        print("An exception occurred at " + str(stations[ii]))
data_Bn = data_Bn[:kk]
data_Be = data_Be[:kk]
data_Bz = data_Bz[:kk]
geo_lat = geo_lat[:kk]
geo_lon = geo_lon[:kk]

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mollweide())

ax.stock_img()
plt.scatter(geo_lon, geo_lat, color="red", s=10, transform=ccrs.PlateCarree())

_ = plt.title("SuperMAG stations")
ax.scatter(0, 0, s=2)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()
# plt.savefig("figures/supermag_stations_locations.png", bbox_inches='tight', dpi=600)

# Triangular Nodes

In [ ]:
mesh = stripy.sTriangulation(geo_lon, geo_lat, refinement_levels=0)

In [ ]:
mesh_lat = mesh.lats * 180 / np.pi
mesh_lon = mesh.lons * 180 / np.pi + 180

## Spherical Elementary Currents (Ridge) Regression 

In [ ]:
# ridge regression regularization
lambda_ = 1e-7
R_earth = 6371  # in km
R_ionosphere = R_earth + 100  # in km

In [ ]:
# specify the secs grid
lat_sec, lon_sec, r_sec = (
    mesh_lat,
    mesh_lon,
    R_ionosphere * np.ones(len(mesh_lat)),
)  # in km

secs_lat_lon_r = np.hstack(
    (lat_sec.reshape(-1, 1), lon_sec.reshape(-1, 1), r_sec.reshape(-1, 1))
)

In [ ]:
# specify the observed grid
obs_lat_lon_r = np.vstack((geo_lat, geo_lon, R_earth * np.ones(len(geo_lon)))).T

In [ ]:
# observations in a vector
B_obs = np.append(np.append(data_Bn, data_Be), data_Bz)
np.shape(B_obs)

In [ ]:
# get T matrix
T_mat = T_df(obs_loc=obs_lat_lon_r, sec_loc=secs_lat_lon_r)
np.shape(T_mat)

In [ ]:
# solve the linear system using Ridge regression
I_sol = ridge_regression(basis_matrix=T_mat, data=B_obs, lambda_=lambda_)

In [ ]:
# create prediction points
n_lat = 300
n_lon = 400
lat_pred, lon_pred, r_pred = np.meshgrid(
    np.linspace(-89, 89, n_lat, endpoint=False),
    np.linspace(0, 360, n_lon, endpoint=False),
    R_earth,
    indexing="ij",
)

pred_lat_lon_r = np.vstack(
    (
        np.ndarray.flatten(lat_pred),
        np.ndarray.flatten(lon_pred),
        np.ndarray.flatten(r_pred),
    )
).T
pred_lat_lon_r.shape

In [ ]:
# predict
B_pred = (
    (T_df(obs_loc=pred_lat_lon_r, sec_loc=secs_lat_lon_r) @ I_sol).reshape((3, -1)).T
)
# reconstruct at stations
B_recon = (
    (T_df(obs_loc=obs_lat_lon_r, sec_loc=secs_lat_lon_r) @ I_sol)
    .reshape((3, len(geo_lat)))
    .T
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), nrows=3, sharex=True)
ax[0].plot(B_recon[:, 0], label="SEC Interpolation", linewidth=2, color="blue")
ax[0].plot(data_Bn, ls=":", label="SuperMAG", linewidth=2, color="red")
ax[0].set_ylabel("Bn [nT]")
_ = ax[0].legend()

ax[1].plot(B_recon[:, 1], label="SEC Interpolation", linewidth=2, color="blue")
ax[1].plot(data_Be, ls=":", label="SuperMAG", linewidth=2, color="red")
ax[1].set_ylabel("Be [nT]")
_ = ax[0].legend()


ax[2].plot(B_recon[:, 2], label="SEC Interpolation", linewidth=2, color="blue")
ax[2].plot(data_Bz, ls=":", label="SuperMAG", linewidth=2, color="red")
ax[2].set_xlabel("observation index")
ax[2].set_ylabel("Bz [nT]")
_ = ax[0].legend()

_ = plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(9, 4))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.coastlines()
pos = ax.pcolormesh(
    lon_pred[:, :, 0],
    lat_pred[:, :, 0],
    B_pred[:, 0].reshape(n_lat, n_lon),
    vmin=-50,
    vmax=50,
    alpha=0.5,
    transform=ccrs.PlateCarree(),
)
plt.scatter(
    geo_lon,
    geo_lat,
    c=data_Bn,
    s=10,
    cmap="viridis",
    vmin=-50,
    vmax=50,
    transform=ccrs.PlateCarree(),
)
cbar = fig.colorbar(pos)
cbar.ax.set_ylabel("$B_{n}$ [nT]", rotation=90)

ax.set_title("SEC interpolation")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_interpolation.png", bbox_inches='tight', dpi=600)

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mollweide())

ax.coastlines()
pos = ax.pcolormesh(
    lon_pred[:, :, 0],
    lat_pred[:, :, 0],
    B_pred[:, 0].reshape(n_lat, n_lon),
    vmin=-50,
    vmax=50,
    alpha=0.4,
    transform=ccrs.PlateCarree(),
)
plt.scatter(
    geo_lon,
    geo_lat,
    c=data_Bn,
    s=10,
    cmap="viridis",
    vmin=-50,
    vmax=50,
    transform=ccrs.PlateCarree(),
)

cbar = fig.colorbar(pos, orientation="horizontal")
cbar.ax.set_ylabel("$B_{n}$ [nT]", rotation=90)


_ = plt.title("$B_{n}$")
ax.scatter(0, 0, s=2)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_locations.png", bbox_inches='tight', dpi=600)

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mollweide())

ax.coastlines()
pos = ax.pcolormesh(
    lon_pred[:, :, 0],
    lat_pred[:, :, 0],
    B_pred[:, 1].reshape(n_lat, n_lon),
    vmin=-50,
    vmax=50,
    alpha=0.4,
    transform=ccrs.PlateCarree(),
)
plt.scatter(
    geo_lon,
    geo_lat,
    c=data_Be,
    s=10,
    cmap="viridis",
    vmin=-50,
    vmax=50,
    transform=ccrs.PlateCarree(),
)

cbar = fig.colorbar(pos, orientation="horizontal")
cbar.ax.set_ylabel("$B_{e}$ [nT]", rotation=90)


_ = plt.title(r"$B_{e}$")
ax.scatter(0, 0, s=2)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_locations.png", bbox_inches='tight', dpi=600)

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Mollweide())

ax.coastlines()
pos = ax.pcolormesh(
    lon_pred[:, :, 0],
    lat_pred[:, :, 0],
    B_pred[:, 2].reshape(n_lat, n_lon),
    vmin=-50,
    vmax=50,
    alpha=0.4,
    transform=ccrs.PlateCarree(),
)
plt.scatter(
    geo_lon,
    geo_lat,
    c=data_Bz,
    s=10,
    cmap="viridis",
    vmin=-50,
    vmax=50,
    transform=ccrs.PlateCarree(),
)

cbar = fig.colorbar(pos, orientation="horizontal")
cbar.ax.set_ylabel("$B_{z}$ [nT]", rotation=90)


_ = plt.title(r"$B_{z}$")
ax.scatter(0, 0, s=2)
ax.set_xlabel("longitude [deg]")
ax.set_ylabel("latitude [deg]")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
plt.tight_layout()

# plt.savefig("figures/supermag_stations_locations.png", bbox_inches='tight', dpi=600)